In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import list_repo_files, hf_hub_download

List file names

In [4]:
files = list_repo_files("Zihan1004/FNSPID", repo_type="dataset")
print(files)

['README.md', 'Stock_news/All_external.csv', 'Stock_news/nasdaq_exteral_data.csv', 'Stock_price/full_history.zip']


In [5]:
base = "hf://datasets/Zihan1004/FNSPID/Stock_news/"
for f in ["All_external.csv", "nasdaq_exteral_data.csv"]:
    df = pd.read_csv(base + f, nrows=2000)
    print(f, df.columns.tolist())
    print(df["Publisher"].value_counts().head(), df["Stock_symbol"].nunique())

All_external.csv ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']
Publisher
Benzinga Staff       438
Lisa Levin           268
Paul Quintaro        197
Benzinga Newsdesk    190
Hal Lindon            69
Name: count, dtype: int64 2
nasdaq_exteral_data.csv ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']
Series([], Name: count, dtype: int64) 5


In [ ]:
t = ["NVDA", "AMD", "INTC", "MSFT", "AAPL"]
f = "Stock_news/All_external.csv"
p = hf_hub_download("Zihan1004/FNSPID", f, repo_type="dataset")

keep = []
for ch in pd.read_csv(p, chunksize=200_000,
                      usecols=["Date", "Article_title",
                               "Stock_symbol", "Publisher"]):
    keep.append(ch[ch["Stock_symbol"].isin(t)])
df = pd.concat(keep)
df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
print(df.groupby(["Stock_symbol", df["Date"].dt.year]).size().unstack())

C:\Users\justi\AppData\Local\Temp\ipykernel_7140\1073679948.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\1073679948.py:6: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,


Date           2011   2012   2013   2014   2015   2016   2017   2018    2019  \
Stock_symbol                                                                   
AAPL            NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN     NaN   
AMD             NaN    NaN    NaN    NaN    NaN    NaN    NaN    NaN    14.0   
INTC            NaN    NaN    NaN    NaN    NaN    NaN    NaN  304.0  1702.0   
NVDA          226.0  183.0  144.0  118.0  180.0  337.0  615.0  526.0   508.0   

Date           2020  
Stock_symbol         
AAPL          473.0  
AMD           236.0  
INTC          469.0  
NVDA          309.0  


In [ ]:
f = "Stock_news/nasdaq_exteral_data.csv"
p = hf_hub_download("Zihan1004/FNSPID", f, repo_type="dataset")

keep = []
for ch in pd.read_csv(p, chunksize=200_000,
                      usecols=["Date", "Article_title",
                               "Stock_symbol", "Publisher"]):
    keep.append(ch[ch["Stock_symbol"].isin(t)])
df = pd.concat(keep)
df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
print(df.groupby(["Stock_symbol", df["Date"].dt.year]).size().unstack())

C:\Users\justi\AppData\Local\Temp\ipykernel_7140\3703766235.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\3703766235.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\3703766235.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\3703766235.py:5: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000,


Date           2011   2012   2013   2014   2015   2016    2017    2018  \
Stock_symbol                                                             
AAPL            NaN    NaN    NaN    NaN    NaN    NaN     NaN     NaN   
AMD             NaN    NaN    NaN    NaN    NaN   34.0  1389.0   926.0   
INTC            NaN    NaN    NaN    NaN    NaN    NaN    26.0  2124.0   
MSFT            NaN    NaN    NaN    NaN    NaN    NaN     NaN     NaN   
NVDA          226.0  183.0  144.0  118.0  180.0  337.0   615.0   526.0   

Date            2019    2020    2021    2022    2023  2024  
Stock_symbol                                                
AAPL             NaN   473.0     NaN  3094.0  5771.0   NaN  
AMD            891.0  1063.0   812.0  1864.0  2160.0  70.0  
INTC          2721.0  1562.0  1212.0  1398.0  2114.0   NaN  
MSFT             NaN     NaN     NaN  2679.0  6058.0   NaN  
NVDA           508.0   309.0   711.0  2368.0  5637.0   NaN  


## Step 1: build the news slice
Combine both files, keep 2018-2023, drop duplicates, save to `data/slice.csv`.

In [ ]:
from pathlib import Path
Path("data").mkdir(exist_ok=True)

cols = ["Date", "Article_title", "Stock_symbol", "Url"]
files = ["Stock_news/All_external.csv",
         "Stock_news/nasdaq_exteral_data.csv"]
parts = []
for f in files:
    p = hf_hub_download("Zihan1004/FNSPID", f, repo_type="dataset")
    for ch in pd.read_csv(p, chunksize=200_000, usecols=cols):
        parts.append(ch[ch["Stock_symbol"].isin(t)].assign(src=f))
news = pd.concat(parts, ignore_index=True)
print("raw rows:", len(news))
print(news.groupby("src")["Date"].apply(lambda s: s.head(3).tolist()))


C:\Users\justi\AppData\Local\Temp\ipykernel_7140\2210655552.py:10: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000, usecols=cols):
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\2210655552.py:10: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000, usecols=cols):
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\2210655552.py:10: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000, usecols=cols):
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\2210655552.py:10: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  for ch in pd.read_csv(p, chunksize=200_000, usecols=cols):
C:\Users\justi\AppData\Local\Temp\ipykernel_7140\2210655552.py:10: DtypeWarning: Columns

raw rows: 56647
src
Stock_news/All_external.csv           [2020-06-10 07:33:26 UTC, 2020-06-10 04:14:08 ...
Stock_news/nasdaq_exteral_data.csv    [2023-12-16 22:00:00 UTC, 2023-12-16 22:00:00 ...
Name: Date, dtype: object


In [9]:
news["Date_raw"] = news["Date"]
news["Date"] = pd.to_datetime(news["Date"], utc=True, errors="coerce")
print("bad dates:", news["Date"].isna().sum())
news = news.dropna(subset=["Date", "Article_title"])
news = news[(news["Date"] >= "2018-01-01") & (news["Date"] < "2024-01-01")]

before = len(news)
news = news.drop_duplicates(subset=["Url", "Stock_symbol"])
news = news.drop_duplicates(
    subset=["Date", "Article_title", "Stock_symbol"])
print("dropped dupes:", before - len(news), "| kept:", len(news))
print(news.groupby(["Stock_symbol", news["Date"].dt.year])
          .size().unstack())
news.to_csv("data/slice.csv", index=False)


bad dates: 0
dropped dupes: 5492 | kept: 46030
Date            2018    2019    2020    2021    2022    2023
Stock_symbol                                                
AAPL             NaN     NaN   472.0     NaN  3018.0  5651.0
AMD            926.0   872.0  1054.0   805.0  1849.0  2127.0
INTC          2040.0  2612.0  1536.0  1185.0  1379.0  2075.0
MSFT             NaN     NaN     NaN     NaN  2606.0  5910.0
NVDA           526.0   507.0   309.0   702.0  2332.0  5537.0


## Step 2: price data
Check which tickers are in `full_history.zip`, then pull SPY.

In [ ]:
import zipfile
zp = hf_hub_download("Zihan1004/FNSPID", "Stock_price/full_history.zip",
                     repo_type="dataset")
z = zipfile.ZipFile(zp)
names = z.namelist()
print(len(names), "files; first 10:", names[:10])
want = [n for n in names if Path(n).stem.upper() in t]
print("our tickers:", want)


d:\projects\Portfolio-triage\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\justi\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


15388 files; first 10: ['full_history/', '__MACOSX/._full_history', 'full_history/RIV.csv', '__MACOSX/full_history/._RIV.csv', 'full_history/ANTE.csv', '__MACOSX/full_history/._ANTE.csv', 'full_history/CSCO.csv', '__MACOSX/full_history/._CSCO.csv', 'full_history/PRI.csv', '__MACOSX/full_history/._PRI.csv']
our tickers: ['full_history/INTC.csv', 'full_history/MSFT.csv', 'full_history/AMD.csv', 'full_history/nvda.csv', 'full_history/AAPL.csv']


d:\projects\Portfolio-triage\.venv\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\justi\.cache\huggingface\hub\datasets--Zihan1004--FNSPID. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [11]:
import yfinance as yf
spy = yf.download("SPY", start="2017-12-01", end="2024-01-10",
                  auto_adjust=False)
spy.columns = spy.columns.get_level_values(0)
spy.to_csv("data/spy.csv")
print(spy.index.min(), spy.index.max(), len(spy))
spy.head()


[*********************100%***********************]  1 of 1 completed

2017-12-01 00:00:00 2024-01-09 00:00:00 1535


Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2017-12-01,230.417694,264.459991,265.309998,260.760010,264.760010,164390900
2017-12-04,230.138855,264.140015,266.799988,264.079987,266.309998,94040600
2017-12-05,229.311127,263.190002,265.149994,263.040009,263.190002,77994500
2017-12-06,229.354691,263.239990,263.730011,262.709991,263.299988,75898600
2017-12-07,230.077881,264.070007,264.429993,262.940002,264.070007,77218600


In [12]:
def peek(name):
    with z.open(name) as fh:
        return pd.read_csv(fh)

for n in want:
    px = peek(n)
    print(n, px.shape, px.columns.tolist())
    print(px.head(2).to_string())
    dc = [c for c in px.columns if "date" in c.lower()][0]
    print("range:", px[dc].min(), "->", px[dc].max())
    print()


full_history/INTC.csv (11040, 7) ['date', 'volume', 'open', 'high', 'low', 'close', 'adj close']
         date    volume       open       high        low      close  adj close
0  2023-12-28  27656100  50.810001  50.869999  50.160000  50.389999  50.389999
1  2023-12-27  52148000  50.630001  51.279999  50.189999  50.759998  50.759998
range: 1980-03-17 -> 2023-12-28

full_history/MSFT.csv (9526, 7) ['date', 'volume', 'open', 'high', 'low', 'close', 'adj close']
         date    volume        open        high         low       close   adj close
0  2023-12-28  14298000  375.369995  376.459991  374.160004  375.279999  375.279999
1  2023-12-27  14905400  373.690002  375.059998  372.809998  374.070007  374.070007
range: 1986-03-13 -> 2023-12-28

full_history/AMD.csv (11040, 7) ['date', 'open', 'high', 'low', 'close', 'adj close', 'volume']
         date        open        high         low       close   adj close    volume
0  2023-12-28  146.800003  150.410004  145.949997  148.759995  148.75999